In [37]:
#Data Collection & Understanding

import os
import pandas as pd

# Absolute Base Directories
INSUREAI_DIR = r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI"
PROJECT_DIR = r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project"

RAW_DIR = os.path.join(INSUREAI_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(INSUREAI_DIR, "data", "processed")

def get_raw_path(filename):
    """Find file in InsureAI/data/raw or fallback locations."""
    path1 = os.path.join(RAW_DIR, filename)
    if os.path.exists(path1):
        return path1
    path2 = os.path.join(PROJECT_DIR, "Insure AI", filename)
    if os.path.exists(path2):
        return path2
    return path1

def clean_insurance_premium():
    print("Processing Medical Insurance dataset...")
    custom_path = get_raw_path("Insurance_Premium_Dataset.xlsx")
    df_custom = pd.read_excel(custom_path)
    
    downloaded_path = os.path.join(PROJECT_DIR, "Insure AI", "insurance.csv")
    if not os.path.exists(downloaded_path):
        downloaded_path = get_raw_path("insurance.csv")
    df_downloaded = pd.read_csv(downloaded_path)
    
    df_downloaded = df_downloaded.rename(columns={'sex': 'gender'})
    df_downloaded['annual_premium_inr'] = (df_downloaded['charges'] * 83).round().astype(int)
    df_downloaded = df_downloaded.drop(columns=['charges'])
    
    region_map = {'southwest': 'west', 'southeast': 'east', 'northwest': 'west', 'northeast': 'east'}
    df_downloaded['region'] = df_downloaded['region'].map(region_map)
    
    df = pd.concat([df_custom, df_downloaded], ignore_index=True)
    os.makedirs(RAW_DIR, exist_ok=True)
    df.to_csv(os.path.join(RAW_DIR, "insurance.csv"), index=False)
    
    df = df.drop_duplicates()
    missing_ids = df['customer_id'].isnull()
    if missing_ids.sum() > 0:
        df.loc[missing_ids, 'customer_id'] = [f"CUST{idx}" for idx in range(200000, 200000 + missing_ids.sum())]
        
    df['bmi'] = df['bmi'].fillna(df['bmi'].median())
    df['annual_income_inr'] = df['annual_income_inr'].fillna(df['annual_income_inr'].median())
    
    for col in ['occupation', 'exercise_frequency', 'alcohol_consumption', 'medical_history', 'family_medical_history']:
        df[col] = df[col].fillna(df[col].mode()[0])
        
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    df.to_csv(os.path.join(PROCESSED_DIR, "insurance_cleaned.csv"), index=False)
    print("Saved cleaned medical insurance dataset.")

def clean_insurance_claims():
    print("Processing Auto Claims dataset...")
    custom_path = get_raw_path("Insurance_Fraud_Claims_Dataset.xlsx")
    df_custom = pd.read_excel(custom_path)
    
    downloaded_path = os.path.join(PROJECT_DIR, "Insure AI", "insurance_claims.csv")
    if not os.path.exists(downloaded_path):
        downloaded_path = get_raw_path("insurance_claims.csv")
    df_downloaded = pd.read_csv(downloaded_path)
    
    df_new = pd.DataFrame()
    df_new['customer_age'] = df_downloaded['age']
    df_new['gender'] = df_downloaded['insured_sex'].str.lower()
    df_new['policy_type'] = 'motor'
    df_new['policy_tenure_years'] = (df_downloaded['months_as_customer'] / 12.0).round(2)
    df_new['annual_premium_inr'] = (df_downloaded['policy_annual_premium'] * 83).round().astype(int)
    df_new['claim_amount_inr'] = (df_downloaded['total_claim_amount'] * 83).round().astype(int)
    
    incident_map = {
        'Single Vehicle Collision': 'collision',
        'Multi-vehicle Collision': 'collision',
        'Vehicle Theft': 'theft',
        'Parked Car': 'third_party_damage'
    }
    df_new['incident_type'] = df_downloaded['incident_type'].map(incident_map).fillna('third_party_damage')
    
    severity_map = {
        'Major Damage': 'major',
        'Minor Damage': 'minor',
        'Total Loss': 'total_loss',
        'Trivial Damage': 'minor'
    }
    df_new['incident_severity'] = df_downloaded['incident_severity'].map(severity_map).fillna('minor')
    
    police_map = {'YES': 'yes', 'NO': 'no'}
    df_new['police_report_filed'] = df_downloaded['police_report_available'].map(police_map).fillna('not_applicable')
    
    df_new['witnesses'] = df_downloaded['witnesses']
    df_new['fraud_reported'] = df_downloaded['fraud_reported']
    
    # Add other custom columns matched to df_custom dtypes
    extra_cols = ['claim_id', 'days_to_report', 'num_past_claims_3yrs', 'documents_complete', 'claim_channel', 'income_bracket', 'claim_location']
    for col in extra_cols:
        df_new[col] = pd.Series(dtype='object' if col not in ['days_to_report', 'num_past_claims_3yrs'] else 'float64')
        
    df_new = df_new[df_custom.columns]
    df = pd.concat([df_custom, df_new], ignore_index=True)
    
    os.makedirs(RAW_DIR, exist_ok=True)
    df.to_csv(os.path.join(RAW_DIR, "insurance_claims.csv"), index=False)
    
    counts = df['fraud_reported'].value_counts()
    pcts = df['fraud_reported'].value_counts(normalize=True) * 100
    print("Class Balance for fraud_reported:")
    for cls in counts.index:
        print(f"  Class {cls}: {counts[cls]} ({pcts[cls]:.2f}%)")
        
    df = df.drop_duplicates()
    missing_ids = df['claim_id'].isnull()
    if missing_ids.sum() > 0:
        df.loc[missing_ids, 'claim_id'] = [f"CLM{idx}" for idx in range(550001, 550001 + missing_ids.sum())]
        
    # Convert numeric columns explicitly to prevent pandas downcasting warnings
    df['days_to_report'] = pd.to_numeric(df['days_to_report'], errors='coerce')
    df['days_to_report'] = df['days_to_report'].fillna(df['days_to_report'].median())
    
    df['num_past_claims_3yrs'] = pd.to_numeric(df['num_past_claims_3yrs'], errors='coerce')
    df['num_past_claims_3yrs'] = df['num_past_claims_3yrs'].fillna(df['num_past_claims_3yrs'].median())
    
    for col in ['documents_complete', 'claim_channel', 'income_bracket', 'claim_location']:
        df[col] = df[col].fillna(df[col].mode()[0])
        
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    df.to_csv(os.path.join(PROCESSED_DIR, "insurance_claims_cleaned.csv"), index=False)
    print("Saved cleaned claims dataset.")

def clean_customer_reviews():
    print("Processing Customer Reviews dataset...")
    reviews_path = get_raw_path("customer_reviews.csv")
    df = pd.read_csv(reviews_path)
    
    df = df.drop_duplicates()
    df = df.dropna(subset=['sentiment', 'model'])
    df['review_text'] = df['review_text'].str.strip()
    
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    df.to_csv(os.path.join(PROCESSED_DIR, "customer_reviews_cleaned.csv"), index=False)
    print("Saved cleaned customer reviews dataset.")

if __name__ == "__main__":
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    clean_insurance_premium()
    clean_insurance_claims()
    clean_customer_reviews()
    print("\nAll datasets cleaned successfully!")

Processing Medical Insurance dataset...
Saved cleaned medical insurance dataset.
Processing Auto Claims dataset...
Class Balance for fraud_reported:
  Class N: 41102 (80.59%)
  Class Y: 9898 (19.41%)
Saved cleaned claims dataset.
Processing Customer Reviews dataset...
Saved cleaned customer reviews dataset.

All datasets cleaned successfully!


In [38]:
import os
import pandas as pd

# Define paths
PROCESSED_DIR = r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\data\processed"

def understand_dataset(filename, target_col):
    filepath = os.path.join(PROCESSED_DIR, filename)
    print("=" * 70)
    print(f" DATASET UNDERSTANDING: {filename}")
    print("=" * 70)
    
    if not os.path.exists(filepath):
        print(f"Error: {filepath} does not exist.")
        return
        
    df = pd.read_csv(filepath)
    
    # 1. Dataset Dimensions & Memory
    print(f"\n1. DIMENSIONS & MEMORY:")
    print(f"   - Total Rows   : {df.shape[0]:,}")
    print(f"   - Total Columns: {df.shape[1]}")
    print(f"   - Memory Usage : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # 2. Data Types & Missing Values
    print(f"\n2. COLUMNS, DATA TYPES & MISSING VALUES:")
    info_df = pd.DataFrame({
        'Data Type': df.dtypes,
        'Missing Values': df.isnull().sum(),
        'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
    })
    print(info_df)
    
    # 3. Sample Data (Head Preview)
    print(f"\n3. SAMPLE DATA (HEAD):")
    print(df.head(3).T)
    
    # 4. Target Variable Analysis
    print(f"\n4. TARGET VARIABLE ({target_col}) ANALYSIS:")
    if target_col in df.columns:
        if pd.api.types.is_numeric_dtype(df[target_col]) and df[target_col].nunique() > 10:
            print(df[target_col].describe().apply(lambda x: f"{x:,.2f}"))
        else:
            counts = df[target_col].value_counts()
            pcts = df[target_col].value_counts(normalize=True) * 100
            target_summary = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts.round(2)})
            print(target_summary)
    print("\n" + "-" * 70 + "\n")

if __name__ == "__main__":
    # Inspect Medical Insurance Premium Dataset
    understand_dataset("insurance_cleaned.csv", target_col="annual_premium_inr")
    
    # Inspect Auto Claims Fraud Dataset
    understand_dataset("insurance_claims_cleaned.csv", target_col="fraud_reported")
    
    # Inspect Customer Reviews Sentiment Dataset
    understand_dataset("customer_reviews_cleaned.csv", target_col="sentiment")


 DATASET UNDERSTANDING: insurance_cleaned.csv

1. DIMENSIONS & MEMORY:
   - Total Rows   : 51,337
   - Total Columns: 14
   - Memory Usage : 26.34 MB

2. COLUMNS, DATA TYPES & MISSING VALUES:
                       Data Type  Missing Values  Missing %
customer_id               object               0        0.0
age                        int64               0        0.0
gender                    object               0        0.0
bmi                      float64               0        0.0
children                   int64               0        0.0
smoker                    object               0        0.0
region                    object               0        0.0
occupation                object               0        0.0
annual_income_inr        float64               0        0.0
exercise_frequency        object               0        0.0
alcohol_consumption       object               0        0.0
medical_history           object               0        0.0
family_medical_history    ob